In [1]:
# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
!pip install -q pandas numpy scikit-learn plotly gdown joblib

import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import gdown
import os
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib
from google.colab import files
# 2. CARGA DE LA BASE DE DATOS DESDE GOOGLE DRIVE (SÓLO SI NO EXISTE)
file_id = '1FRQ53-xpLjLawU84eokdKu_f-4msRsE8'
url = f'https://drive.google.com/uc?id={file_id}'
db_filename = 'agrotech.db'

if not os.path.exists(db_filename):
    print("📥 Descargando la base de datos desde Google Drive...")
    gdown.download(url, db_filename, quiet=False)
else:
    print("✅ El archivo 'agrotech.db' ya existe localmente. No se sobrescribirá para preservar tus nuevos datos.")

📥 Descargando la base de datos desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1FRQ53-xpLjLawU84eokdKu_f-4msRsE8
To: /content/agrotech.db
100%|██████████| 258k/258k [00:00<00:00, 34.4MB/s]


In [2]:
# 3. EXTRACCIÓN Y PREPARACIÓN DE DATOS CON VARIABLES CÍCLICAS TEMPORALES
import numpy as np

conn = sqlite3.connect(db_filename)
query = """
    SELECT timestamp, nodo_id, temperatura, humedad,
           humedad_suelo, temperatura_suelo, ph, conductividad,
           nitrogeno, fosforo, potasio
    FROM mediciones
    WHERE temperatura IS NOT NULL AND humedad_suelo IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
conn.close()

if len(df) < 50:
    print(f"⚠️ Advertencia: Tienes {len(df)} registros. El modelo requiere mínimo 50 para resultados fiables.")

# Limpieza de datos básica
df_clean = df.dropna().copy()

# Convertir timestamp a tipo datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])

# Extraer componentes de tiempo numéricos
df_clean['hora'] = df_clean['timestamp'].dt.hour
df_clean['mes'] = df_clean['timestamp'].dt.month

# Aplicar transformaciones trigonométricas cíclicas
df_clean['hora_sin'] = np.sin(2 * np.pi * df_clean['hora'] / 24.0)
df_clean['hora_cos'] = np.cos(2 * np.pi * df_clean['hora'] / 24.0)
df_clean['mes_sin'] = np.sin(2 * np.pi * df_clean['mes'] / 12.0)
df_clean['mes_cos'] = np.cos(2 * np.pi * df_clean['mes'] / 12.0)

print("✅ Variables temporales cíclicas creadas correctamente.")

✅ Variables temporales cíclicas creadas correctamente.


```markdown
## 4. Agrupación por Dominios Cohesivos y Reducción de Dimensionalidad (PCA)
Dividiremos las variables en 3 grupos de relevancia física y química:
- **Nutrientes (Nutrients Group):** `nitrogeno`, `fosforo`, `potasio`, `ph`, `conductividad`
- **Suelo (Soil Group):** `humedad_suelo`, `temperatura_suelo`
- **Ambiente (Ambient Group):** `temperatura`, `humedad`, `hora_sin`, `hora_cos`, `mes_sin`, `mes_cos`

Aplicaremos PCA a cada grupo individualmente para obtener una representación simplificada pero potente, y luego uniremos estas 3 componentes para entrenar los modelos (Isolation Forest y Random Forest). Ningún modelo recibe las 13 variables originales.
```

In [3]:
# Definición de los 3 grupos de variables (Agregamos las variables cíclicas al grupo de ambiente)
grupo_nutrientes = ['nitrogeno', 'fosforo', 'potasio', 'ph', 'conductividad']
grupo_suelo = ['humedad_suelo', 'temperatura_suelo']
grupo_ambiente = ['temperatura', 'humedad', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos']

# 1. Normalización por separado de cada grupo
scaler_nutrientes = StandardScaler()
scaler_suelo = StandardScaler()
scaler_ambiente = StandardScaler()

scaled_nut = scaler_nutrientes.fit_transform(df_clean[grupo_nutrientes])
scaled_suelo = scaler_suelo.fit_transform(df_clean[grupo_suelo])
scaled_amb = scaler_ambiente.fit_transform(df_clean[grupo_ambiente])

# Reducción de dimensionalidad a 1 componente principal por grupo
pca_nutrientes = PCA(n_components=1, random_state=42)
pca_suelo = PCA(n_components=1, random_state=42)
pca_ambiente = PCA(n_components=1, random_state=42)

# Extraemos las representaciones reducidas
comp_nut = pca_nutrientes.fit_transform(scaled_nut)
comp_suelo = pca_suelo.fit_transform(scaled_suelo)
comp_amb = pca_ambiente.fit_transform(scaled_amb)

# Creamos un nuevo dataframe con las componentes consolidadas
X_grouped_pca = pd.DataFrame({
    'Comp_Nutrientes': comp_nut.ravel(),
    'Comp_Suelo': comp_suelo.ravel(),
    'Comp_Ambiente': comp_amb.ravel()
}, index=df_clean.index)

# Las 3 componentes son las ÚNICAS entradas que reciben los modelos (Isolation Forest y Random Forest)
features_pca = ['Comp_Nutrientes', 'Comp_Suelo', 'Comp_Ambiente']

display(X_grouped_pca.head())

,Comp_Nutrientes,Comp_Suelo,Comp_Ambiente
0,-0.351263,-1.002058,-1.232659
1,-0.921874,-0.281936,-2.110709
2,-1.298460,-1.082825,0.777773
3,-1.298460,-1.082825,0.777773
4,-1.275838,-0.266629,-1.388032


```markdown
## 5. Entrenamiento del Modelo Único de Detección de Anomalías
Entrenamos un único `IsolationForest` utilizando las tres variables consolidadas que representan cada dominio.
```

In [4]:

# Entrenamiento del modelo consolidador
modelo_unico = IsolationForest(
    n_estimators=500,
    contamination=0.5,
    random_state=42
)

modelo_unico.fit(X_grouped_pca)

# Predicciones básicas
df_clean['Score_Consolidado'] = modelo_unico.decision_function(X_grouped_pca)
df_clean['Pred_Consolidado'] = modelo_unico.predict(X_grouped_pca)

```markdown
## 6. Diagnóstico de Causa Raíz de la Anomalía
¿Cómo sabemos a nivel de salida por qué falló?
Calcularemos la desviación absoluta estandarizada de cada grupo respecto al comportamiento normal (mediana) de los datos catalogados como "Sanos". El grupo que presente la mayor desviación será asignado como la causa de la anomalía.
```

In [5]:
# Agregamos primero los componentes al df_clean para asegurar una alineación de índices segura
df_clean['Comp_Nutrientes'] = X_grouped_pca['Comp_Nutrientes']
df_clean['Comp_Suelo'] = X_grouped_pca['Comp_Suelo']
df_clean['Comp_Ambiente'] = X_grouped_pca['Comp_Ambiente']

# Obtenemos los valores de referencia (medianas de los casos sanos) usando las columnas de Pandas alineadas
sanos_df = df_clean[df_clean['Pred_Consolidado'] == 1]
mediana_nut = sanos_df['Comp_Nutrientes'].median()
mediana_suelo = sanos_df['Comp_Suelo'].median()
mediana_amb = sanos_df['Comp_Ambiente'].median()

def diagnosticar_anomalia(row):
    if row['Pred_Consolidado'] == 1:
        return 'Sin anomalías (Sana)'

    # Calculamos la desviación absoluta respecto a los valores normales
    dev_nut = abs(row['Comp_Nutrientes'] - mediana_nut)
    dev_suelo = abs(row['Comp_Suelo'] - mediana_suelo)
    dev_amb = abs(row['Comp_Ambiente'] - mediana_amb)

    desviaciones = {
        'Anomalía de Nutrientes': dev_nut,
        'Anomalía de Suelo': dev_suelo,
        'Anomalía de Ambiente': dev_amb
    }

    # La causa raíz será la que presente mayor desviación
    return max(desviaciones, key=desviaciones.get)

# Aplicamos el diagnóstico
df_clean['Diagnostico'] = df_clean.apply(diagnosticar_anomalia, axis=1)

# Mostrar el resumen de los diagnósticos de anomalías detectadas
anomalias_detectadas = df_clean[df_clean['Pred_Consolidado'] == -1]
print(f"Total de anomalías detectadas: {len(anomalias_detectadas)}")
display(anomalias_detectadas[['timestamp', 'nodo_id', 'Score_Consolidado', 'Diagnostico']].head(10))

Total de anomalías detectadas: 120


,timestamp,nodo_id,Score_Consolidado,Diagnostico
0,2024-07-30 12:01:00,nodo1,-0.041767,Anomalía de Ambiente
1,2024-07-30 15:00:00,nodo1,-0.051967,Anomalía de Ambiente
5,2024-07-31 15:01:00,nodo1,-0.143463,Anomalía de Ambiente
11,2024-08-02 15:01:00,nodo1,-0.005128,Anomalía de Ambiente
17,2024-08-05 09:01:00,nodo1,-0.132804,Anomalía de Nutrientes
18,2024-08-05 12:01:00,nodo1,-0.084984,Anomalía de Nutrientes
19,2024-08-05 15:01:00,nodo1,-0.079556,Anomalía de Nutrientes
20,2024-08-06 09:01:00,nodo1,-0.052707,Anomalía de Nutrientes
21,2024-08-06 12:01:00,nodo1,-0.012458,Anomalía de Nutrientes
25,2024-08-07 15:01:00,nodo1,-0.038639,Anomalía de Suelo


```markdown
## 7. Visualización 3D Interactiva del Espacio de Anomalías Agrupadas
Graficamos las 3 dimensiones simplificadas (Nutrientes, Suelo y Ambiente) coloreando los puntos según su diagnóstico final.
```

In [6]:

# Renderizado del gráfico 3D usando las nuevas columnas consolidadas
fig_diagnosticos = px.scatter_3d(
    df_clean,
    x='Comp_Nutrientes',
    y='Comp_Suelo',
    z='Comp_Ambiente',
    color='Diagnostico',
    color_discrete_map={
        'Sin anomalías (Sana)': '#2E7D32',
        'Anomalía de Nutrientes': '#C62828',
        'Anomalía de Suelo': '#F57F17',
        'Anomalía de Ambiente': '#1565C0'
    },
    title="Espacio del Modelo de 3 Dominios (Nutrientes vs Suelo vs Ambiente)",
    hover_data=['temperatura', 'humedad', 'humedad_suelo', 'temperatura_suelo',
    'ph', 'conductividad', 'nitrogeno', 'fosforo', 'potasio'],
    labels={
        'Comp_Nutrientes': 'Nutrientes',
        'Comp_Suelo': 'Suelo',
        'Comp_Ambiente': 'Ambiente'}
)
fig_diagnosticos.update_traces(marker=dict(size=6, opacity=0.8))
fig_diagnosticos.show()

In [7]:

# Guardamos el modelo unificado
joblib.dump(modelo_unico, 'iforest_model.joblib')
print("✅ Nuevo modelo simplificado con agrupación por PCA exportado como 'iforest_model.joblib'")

✅ Nuevo modelo simplificado con agrupación por PCA exportado como 'iforest_model.joblib'


In [8]:
# Resumen del modelo de PCA agrupado consolidado
print("📊 Resumen de Clasificación del Modelo de PCA Agrupado:")
if 'Diagnostico' in df_clean.columns:
    print(df_clean['Diagnostico'].value_counts())
else:
    print("⚠️ Por favor, ejecuta la celda 78f116f1 para volver a generar los diagnósticos en el DataFrame actual.")

📊 Resumen de Clasificación del Modelo de PCA Agrupado:
Diagnostico
Sin anomalías (Sana)      120
Anomalía de Ambiente       48
Anomalía de Nutrientes     36
Anomalía de Suelo          36
Name: count, dtype: int64


In [9]:
# Exportar únicamente el modelo consolidado basado en PCA
joblib.dump(modelo_unico, 'iforest_model_pca.joblib')
print("✅ Modelo basado en PCA exportado correctamente como 'iforest_model_pca.joblib'.")

✅ Modelo basado en PCA exportado correctamente como 'iforest_model_pca.joblib'.


```markdown
## 8. Experimento: Clasificación Supervisada con Random Forest

Vamos a utilizar las etiquetas de anomalía que obtuvimos mediante nuestro diagnóstico heurístico como etiquetas de entrenamiento para un clasificador **Random Forest**.

Usaremos las **3 componentes PCA** (`Comp_Nutrientes`, `Comp_Suelo`, `Comp_Ambiente`) en lugar de las variables originales, de modo que el Random Forest reciba la misma representación reducida que el Isolation Forest, y evaluar qué tan bien puede aprender a clasificar estos dominios automáticamente.
```

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd
import plotly.express as px

# 1. Variables originales (13D: sensores + temporales cíclicas). Ya NO se entregan al modelo:
#    son la entrada del PCA y se conservan para el hover de los gráficos.
features_originales = [
    'nitrogeno', 'fosforo', 'potasio', 'ph', 'conductividad',
    'humedad_suelo', 'temperatura_suelo', 'temperatura', 'humedad',
    'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos'
]

# Variables predictoras del Random Forest: las 3 componentes PCA
X = df_clean[features_pca]
y = df_clean['Diagnostico']

# 2. División en conjuntos de entrenamiento y prueba (Stratified para mantener la proporción de anomalías)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Inicialización y entrenamiento del Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Predicciones
y_pred = rf_model.predict(X_test)

# 5. Evaluación del modelo
print("📊 Reporte de Clasificación del Random Forest (Entrada: 3 componentes PCA):")
print(classification_report(y_test, y_pred, zero_division=0))

# 6. Importancia de las variables reales en la clasificación
importancia = pd.Series(rf_model.feature_importances_, index=features_pca).sort_values(ascending=False)

fig_importance = px.bar(
    importancia,
    x=importancia.values,
    y=importancia.index,
    orientation='h',
    title='Importancia de las Componentes PCA (Random Forest)',
    labels={'x': 'Importancia (Gini)', 'y': 'Variable'}
)
fig_importance.show()

📊 Reporte de Clasificación del Random Forest (Entrada: 3 componentes PCA):
                        precision    recall  f1-score   support

  Anomalía de Ambiente       1.00      1.00      1.00        14
Anomalía de Nutrientes       1.00      0.91      0.95        11
     Anomalía de Suelo       1.00      0.55      0.71        11
  Sin anomalías (Sana)       0.86      1.00      0.92        36

              accuracy                           0.92        72
             macro avg       0.96      0.86      0.90        72
          weighted avg       0.93      0.92      0.91        72



```markdown
## 9. Clasificación Determinista (Cultivo de Tomate) y Comparación Visual con Random Forest

Primero, definimos las reglas de negocio agronómicas para el tomate y etiquetamos los datos de manera determinista.
```

In [11]:
# Rangos óptimos para el cultivo de tomate
RANGES_SANA = {
    "temperatura": (18.0, 32.0),
    "humedad": (45.0, 85.0),
    "nitrogeno": (5, 50),
    "fosforo": (5, 50),
    "potasio": (5, 250),
    "humedad_suelo": (15.0, 50.0),  #para suelo arcilloso
    "temperatura_suelo": (15.0, 30.0),
    "conductividad": (50, 400), #¿huh?
    "ph": (5.5, 6.8),
}

def etiquetar_agronomicamente(row):
    # 1. Verificar anomalías en el subgrupo de Ambiente
    temp_fuera = row['temperatura'] < RANGES_SANA['temperatura'][0] or row['temperatura'] > RANGES_SANA['temperatura'][1]
    hum_fuera = row['humedad'] < RANGES_SANA['humedad'][0] or row['humedad'] > RANGES_SANA['humedad'][1]

    # 2. Verificar anomalías en el subgrupo de Suelo
    h_suelo_fuera = row['humedad_suelo'] < RANGES_SANA['humedad_suelo'][0] or row['humedad_suelo'] > RANGES_SANA['humedad_suelo'][1]
    t_suelo_fuera = row['temperatura_suelo'] < RANGES_SANA['temperatura_suelo'][0] or row['temperatura_suelo'] > RANGES_SANA['temperatura_suelo'][1]

    # 3. Verificar anomalías en el subgrupo de Nutrientes / Químico
    n_fuera = row['nitrogeno'] < RANGES_SANA['nitrogeno'][0] or row['nitrogeno'] > RANGES_SANA['nitrogeno'][1]
    p_fuera = row['fosforo'] < RANGES_SANA['fosforo'][0] or row['fosforo'] > RANGES_SANA['fosforo'][1]
    k_fuera = row['potasio'] < RANGES_SANA['potasio'][0] or row['potasio'] > RANGES_SANA['potasio'][1]
    ph_fuera = row['ph'] < RANGES_SANA['ph'][0] or row['ph'] > RANGES_SANA['ph'][1]
    ec_fuera = row['conductividad'] < RANGES_SANA['conductividad'][0] or row['conductividad'] > RANGES_SANA['conductividad'][1]

    # Prioridad de asignación si hay múltiples fallos (basada en la severidad física)
    if n_fuera or p_fuera or k_fuera or ph_fuera or ec_fuera:
        return 'Anomalía de Nutrientes'
    elif h_suelo_fuera or t_suelo_fuera:
        return 'Anomalía de Suelo'
    elif temp_fuera or hum_fuera:
        return 'Anomalía de Ambiente'
    else:
        return 'Sin anomalías (Sana)'

# Aplicamos la categorización real agronómica
df_clean['Etiqueta_Agronomica'] = df_clean.apply(etiquetar_agronomicamente, axis=1)
print("📊 Distribución de registros con la nueva regla determinista de Tomate:")
print(df_clean['Etiqueta_Agronomica'].value_counts())

📊 Distribución de registros con la nueva regla determinista de Tomate:
Etiqueta_Agronomica
Anomalía de Nutrientes    137
Sin anomalías (Sana)       53
Anomalía de Suelo          45
Anomalía de Ambiente        5
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Entrenamos el Random Forest con las etiquetas realistas de tomate usando las 3 componentes PCA
# (el grupo Ambiente del PCA ya incorpora las variables temporales cíclicas)
X_rf = df_clean[features_pca]
y_rf = df_clean['Etiqueta_Agronomica']

try:
    X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
        X_rf, y_rf, test_size=0.3, random_state=42
    )
    rf_tomate = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_tomate.fit(X_train_rf, y_train_rf)
    print("📈 Reporte de Clasificación con Reglas Agronómicas reales (Test Set, entrada: 3 componentes PCA):")
    print(classification_report(y_test_rf, rf_tomate.predict(X_test_rf), zero_division=0))
except Exception as e:
    print(f"⚠️ Ocurrió un inconveniente con el Split: {e}. Entrenando con el dataset completo para fines de mapeo visual.")
    rf_tomate = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_tomate.fit(X_rf, y_rf)

# Guardamos las predicciones totales del modelo en nuestro DataFrame
df_clean['Pred_RandomForest'] = rf_tomate.predict(X_rf)
print("✅ Clasificaciones de Random Forest (componentes PCA) guardadas correctamente.")

📈 Reporte de Clasificación con Reglas Agronómicas reales (Test Set, entrada: 3 componentes PCA):
                        precision    recall  f1-score   support

  Anomalía de Ambiente       0.00      0.00      0.00         1
Anomalía de Nutrientes       0.97      0.88      0.92        41
     Anomalía de Suelo       0.76      1.00      0.87        13
  Sin anomalías (Sana)       0.83      0.88      0.86        17

              accuracy                           0.89        72
             macro avg       0.64      0.69      0.66        72
          weighted avg       0.89      0.89      0.88        72

✅ Clasificaciones de Random Forest (componentes PCA) guardadas correctamente.


```markdown
### 10. Visualización Comparativa 3D (Espacio PCA - Clasificación por Random Forest)

Ahora, utilizaremos los mismos componentes de PCA creados anteriormente para proyectar visualmente la clasificación predictiva realizada por el **Random Forest**.
```

In [13]:
import plotly.express as px

fig_rf_3d = px.scatter_3d(
    df_clean,
    x='Comp_Ambiente',
    y='Comp_Nutrientes',
    z='Comp_Suelo',
    color='Pred_RandomForest',
    color_discrete_map={
        'Sin anomalías (Sana)': '#2E7D32',
        'Anomalía de Nutrientes': '#F57F17',
        'Anomalía de Suelo': '#80430e',
        'Anomalía de Ambiente': '#1565C0'
    },
    title="Clasificación Agronómica de Tomate con Random Forest (Espacio PCA)",
    hover_data=['timestamp'] + features_originales,
    labels={
        'Comp_Nutrientes': 'Componente Nutrientes',
        'Comp_Suelo': 'Componente Suelo',
        'Comp_Ambiente': 'Componente Ambiente'
    }
)
fig_rf_3d.update_traces(marker=dict(size=6, opacity=0.8))
fig_rf_3d.show()

In [15]:
# Guardar el modelo entrenado de Random Forest con reglas agronómicas
import joblib
from google.colab import files

# El RF ahora recibe las 3 componentes PCA, por lo que se exporta JUNTO con los escaladores y PCA
# ajustados en la sección 4 (los mismos con los que se entrenó). Formato compatible con la app AgroTech.
rf_pack = {
    'rf': rf_tomate,
    'scaler_nut': scaler_nutrientes,
    'scaler_suelo': scaler_suelo,
    'scaler_amb': scaler_ambiente,
    'pca_nut': pca_nutrientes,
    'pca_suelo': pca_suelo,
    'pca_amb': pca_ambiente,
    'features': features_originales,   # 13 características originales (entrada de la app)
    'features_pca': features_pca,      # 3 componentes (entrada real del modelo)
    'grupo_nutrientes': grupo_nutrientes,
    'grupo_suelo': grupo_suelo,
    'grupo_ambiente': grupo_ambiente
}

# Exportar el paquete
joblib.dump(rf_pack, 'rf_model_pca.joblib')
print("✅ Modelo Random Forest de Tomate (PCA 3D) exportado correctamente como 'rf_model_pca.joblib'.")

# Descargar el archivo automáticamente
#files.download('rf_model_pca.joblib')

✅ Modelo Random Forest de Tomate (PCA 3D) exportado correctamente como 'rf_model_pca.joblib'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>